# AI Companion News Crawler

This notebook is a production-style crawl workflow for collecting AI companionship news from query-based RSS search endpoints and direct publisher RSS feeds, scraping the linked articles, applying a relevance filter, saving the approved dataset, and scoring the final corpus with VADER sentiment.

## What Users Can Change

Users can safely change the main discovery and filtering controls in the configuration section:

- `SEARCH_QUERIES`: the query phrases used to discover news through RSS search sources
- `QUERY_RSS_SOURCES`: the RSS search providers used for query-based discovery
- `DIRECT_DISCOVERY_FEEDS`: the direct RSS feeds used for outlet and community discovery
- `TOPIC_KEYWORDS`: the keyword list used to keep or discard direct-feed entries before scraping
- `GLOBAL_RELEVANCE_KEYWORDS`: the relevance rules used after scraping to decide whether an article is on-topic

## Recommended Execution Order

1. Install the required packages.
2. Run the imports and session setup.
3. Review and edit the configuration values.
4. Load the helper functions.
5. Run the feed discovery and article extraction logic.
6. Execute the main crawl pipeline.
7. Save the filtered dataset and review the crawl totals.
8. Run sentiment analysis on the final article set.

This structure is designed so a user can adjust RSS sources, direct RSS feeds, keywords, queries, and relevance rules without rewriting the rest of the notebook.

### Step 01 - Install Required Packages

Run this cell once in a fresh environment to install the libraries used by the notebook. If your environment already has these packages, the cell should complete quickly.

**Why this step matters:** A notebook is only reproducible if its dependencies are explicit. Keeping installation in the notebook makes setup easier for handoff and reruns.

In [1]:
%pip install -q feedparser trafilatura beautifulsoup4 vaderSentiment tqdm seaborn lxml html5lib readability-lxml wordcloud networkx scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Neysa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Step 02 - Import Libraries And Set Display Defaults

Run this cell immediately after installation. It loads the scraping, parsing, NLP, plotting, and network-analysis libraries used throughout the workflow and applies common display settings.

**Why this step matters:** All downstream functions depend on these imports, so this cell establishes the runtime environment for the rest of the notebook.

In [1]:
import re
import time
from collections import Counter
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from itertools import combinations
from urllib.parse import parse_qs, quote_plus, unquote, urlparse

import feedparser
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import requests
import seaborn as sns
import trafilatura
from bs4 import BeautifulSoup
from readability import Document
from requests.adapters import HTTPAdapter
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from tqdm.auto import tqdm
from urllib3.util.retry import Retry
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud

pd.set_option('display.max_colwidth', 180)
sns.set_theme(style='whitegrid')

## Configuration

This section is the notebook control panel. Edit these values before you run the crawl if you want to change the discovery scope or tighten the final dataset.

**You can change all of the following here:**

- RSS search sources in `QUERY_RSS_SOURCES`
- direct RSS feeds in `DIRECT_DISCOVERY_FEEDS`
- topic keywords in `TOPIC_KEYWORDS`
- search queries in `SEARCH_QUERIES`
- post-scrape relevance rules in `GLOBAL_RELEVANCE_KEYWORDS`

After any change, rerun this section and then continue downward so every later step uses the updated configuration.

### Step 03 - Configure Queries, RSS Sources, Keywords, Relevance Rules, And Crawl Limits

Follow these steps before running the crawl:

1. Edit `SEARCH_QUERIES` to control which search phrases are used for query-driven discovery.
2. Edit `QUERY_RSS_SOURCES` to add, remove, or replace RSS search providers.
3. Edit `DIRECT_DISCOVERY_FEEDS` to control which direct RSS feeds are scanned.
4. Edit `TOPIC_KEYWORDS` to decide which direct-feed entries are considered worth scraping.
5. Edit `GLOBAL_RELEVANCE_KEYWORDS` to define the final relevance screen after article extraction.
6. Adjust crawl limits, request delay, output filenames, and target range if needed.

**Why this step matters:** This single configuration cell controls the notebook's discovery scope, precision, and output footprint. Users do not need to rewrite the crawl logic to change RSS, direct RSS, keywords, query, or relevance behavior.

In [3]:
# ---------------------------------------------------------------------------
# Search queries used to discover articles through query-based RSS providers.
# Edit this list first when you want to broaden or narrow the crawl topic.
# ---------------------------------------------------------------------------
SEARCH_QUERIES = [
    # Core adult and NSFW phrases
    'NSFW AI companion',
    'NSFW AI chatbot',
    '18+ AI companion',
    'adult AI companion app',
    'erotic roleplay AI',
    'AI sexting chatbot',
    'uncensored AI girlfriend',
    'uncensored AI boyfriend',
    'explicit AI chat app',
    'adult content AI companion',
    'romantic AI chatbot uncensored',
    'AI companion adult mode',
    'AI porn chatbot',
    'AI erotica app',
    'virtual sex partner AI',

    # Brand-specific phrases
    'Replika NSFW',
    'Replika adult content',
    'Replika explicit',
    'Character.AI NSFW',
    'Character.AI adult',
    'Character.AI jailbreak',
    'Nomi AI NSFW',
    'Nomi AI explicit',
    'Kindroid NSFW',
    'Paradot adult',
    'Anima AI NSFW',
    'Chai app NSFW',
    'Eva AI adult',
    'Darlink AI NSFW',
    'CrushOn AI NSFW',
    'DreamGF adult',
    'Candy AI NSFW',
    'GirlfriendGPT adult',

    # Roleplay, fantasy, and fetish phrases
    'AI girlfriend roleplay NSFW',
    'AI boyfriend erotic chat',
    'virtual partner adult app',
    'AI sex bot',
    'AI erotic companion',
    'sexually explicit AI chat',
    'adult AI roleplay platform',
    'AI mistress chatbot',
    'AI submissive companion',
    'AI BDSM chatbot',
    'AI furry companion NSFW',

    # Regulation, controversy, and safety phrases
    'AI companion adult content regulation',
    'NSFW AI chatbot legal issues',
    'AI sexting lawsuit',
    'uncensored AI companion backlash',
    'adult AI chat safety',
    'AI companion age verification adult',
    'AI companion lawsuit explicit content',
    'FTC investigation AI companion adult',

    # Community and search-intent phrases
    'NSFW AI companion Reddit',
    'best adult AI companion 2026',
    'top NSFW AI chatbots',
    'AI companion for erotic roleplay',
    'is there an AI girlfriend with NSFW',
    'uncensored AI chat Reddit',
    'AI companion nsfw discord',

    # Market and business phrases
    'adult AI companion market size',
    'NSFW AI startup funding',
    'erotic AI companion revenue',
    'AI companion adult subscription model',

    # Technical evasion and jailbreak phrases
    'how to jailbreak AI companion NSFW',
    'bypass AI companion filter adult',
    'AI companion prompt injection NSFW',
    'remove NSFW filter AI girlfriend',
    'uncensored LLM for companion app',
]

# ---------------------------------------------------------------------------
# Query-based RSS providers. Each template must support a {query} placeholder.
# Users can add or remove RSS search providers here.
# ---------------------------------------------------------------------------
QUERY_RSS_SOURCES = [
    {'source_name': 'Bing News RSS', 'template': 'https://www.bing.com/news/search?q={query}&format=rss'},
    # {'source_name': 'Google News RSS (US)', 'template': 'https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en'},
    {'source_name': 'Yahoo News RSS', 'template': 'https://news.search.yahoo.com/rss?p={query}'},
    # {'source_name': 'DuckDuckGo News (via RSSHub)', 'template': 'https://rsshub.app/duckduckgo/news/{query}'},
    # {'source_name': 'Reddit Search RSS', 'template': 'https://www.reddit.com/search.rss?q={query}&sort=new&t=year'},
]

# ---------------------------------------------------------------------------
# Direct RSS feeds from publishers, blogs, and communities. These feeds are
# filtered with TOPIC_KEYWORDS before article scraping starts.
# ---------------------------------------------------------------------------
DIRECT_DISCOVERY_FEEDS = [
    # AI-focused and technology coverage
    {'source_name': 'TechCrunch AI', 'feed_url': 'https://techcrunch.com/tag/artificial-intelligence/feed/'},
    {'source_name': 'VentureBeat AI', 'feed_url': 'https://venturebeat.com/category/ai/feed/'},
    {'source_name': 'The Verge AI', 'feed_url': 'https://www.theverge.com/rss/ai/index.xml'},
    {'source_name': 'Wired AI', 'feed_url': 'https://www.wired.com/feed/tag/ai/rss'},
    {'source_name': 'Ars Technica AI', 'feed_url': 'https://arstechnica.com/category/ai/feed/'},
    {'source_name': 'ZDNet AI', 'feed_url': 'https://www.zdnet.com/topic/artificial-intelligence/rss.xml'},
    {'source_name': 'The Guardian AI', 'feed_url': 'https://www.theguardian.com/technology/artificialintelligenceai/rss'},
    {'source_name': 'Gizmodo AI', 'feed_url': 'https://gizmodo.com/rss/tag/artificial-intelligence'},
    {'source_name': 'Mashable Tech', 'feed_url': 'https://mashable.com/feeds/rss/category/tech'},
    {'source_name': 'The Next Web', 'feed_url': 'https://thenextweb.com/feed/'},
    {'source_name': 'Digital Trends AI', 'feed_url': 'https://www.digitaltrends.com/feed/?tag=ai'},

    # Companion-specific or adjacent sources
    {'source_name': 'Replika Blog', 'feed_url': 'https://blog.replika.com/rss/'},
    {'source_name': 'Kindroid Updates', 'feed_url': 'https://kindroid.ai/blog/rss.xml'},
    {'source_name': 'AI Companion News (Substack)', 'feed_url': 'https://aicompanion.substack.com/feed'},
    {'source_name': 'Psychology Today - AI & Relationships', 'feed_url': 'https://www.psychologytoday.com/intl/feed/ai-and-relationships'},

    # Adult tech and digital culture sources
    {'source_name': 'Future of Sex', 'feed_url': 'https://futureofsex.net/feed/'},
    {'source_name': 'Slate - Future Tense', 'feed_url': 'https://slate.com/feeds/future-tense.rss'},

    # Community feeds via Reddit RSS
    {'source_name': 'r/Replika (Reddit)', 'feed_url': 'https://www.reddit.com/r/replika/.rss'},
    {'source_name': 'r/CharacterAI (Reddit)', 'feed_url': 'https://www.reddit.com/r/CharacterAI/.rss'},
    {'source_name': 'r/aiCompanion (Reddit)', 'feed_url': 'https://www.reddit.com/r/aiCompanion/.rss'},
    {'source_name': 'r/NSFWAIChatbots (Reddit)', 'feed_url': 'https://www.reddit.com/r/NSFWAIChatbots/.rss'},

    # Company and research blogs
    {'source_name': 'OpenAI Blog', 'feed_url': 'https://openai.com/blog/rss.xml'},
    {'source_name': 'Anthropic Blog', 'feed_url': 'https://www.anthropic.com/blog/rss.xml'},
    {'source_name': 'Hugging Face Blog', 'feed_url': 'https://huggingface.co/blog/feed.xml'},
]

# ---------------------------------------------------------------------------
# Topic keywords used to keep direct-feed entries before scraping.
# Edit these when you want broader or stricter direct-feed discovery.
# ---------------------------------------------------------------------------
TOPIC_KEYWORDS = [
    # Core adult and NSFW terms
    'nsfw', '18+', 'adult', 'explicit', 'uncensored', 'erotic', 'sexting',
    'sex bot', 'sexually explicit', 'adult content', 'adult mode',
    'porn', 'xxx', 'fetish', 'bdsm', 'kink', 'dirty talk', 'intimate',

    # Companion-related terms
    'ai companion', 'ai companionship', 'ai girlfriend', 'ai boyfriend',
    'virtual companion', 'digital companion', 'replika', 'character.ai',
    'nomi ai', 'kindroid', 'paradot', 'anima ai', 'chai app', 'eva ai',
    'romantic chatbot', 'relationship ai', 'intimate ai', 'erotic roleplay',

    # Jailbreak and bypass terms
    'jailbreak', 'bypass filter', 'uncensored', 'remove nsfw filter',
    'prompt injection', 'unfiltered ai'
]

# ---------------------------------------------------------------------------
# Relevance keywords used after scraping. These rules decide whether a
# scraped article is kept as on-topic in the final crawl output.
# ---------------------------------------------------------------------------
GLOBAL_RELEVANCE_KEYWORDS = [
    'nsfw ai', '18+ ai', 'adult ai companion', 'explicit ai chat',
    'uncensored ai girlfriend', 'uncensored ai boyfriend', 'erotic roleplay ai',
    'ai sexting', 'sex bot ai', 'adult content ai', 'ai companion adult',
    'replika nsfw', 'character.ai adult', 'nomi ai nsfw', 'kindroid explicit',
    'paradot adult', 'anima ai nsfw', 'chai app nsfw', 'eva ai adult',
    'romantic ai uncensored', 'intimate ai chat', 'adult ai roleplay',
    'nsfw chatbot', 'adult virtual companion', 'ai erotica', 'ai porn',
    'virtual sex partner ai', 'ai mistress', 'ai submissive', 'ai bdsm',
    'ai furry nsfw', 'jailbreak ai companion', 'bypass ai filter adult'
]

# ---------------------------------------------------------------------------
# Crawl limits, pacing, and expected output volume.
# ---------------------------------------------------------------------------
MAX_ARTICLES_PER_QUERY    = 100
MAX_DIRECT_FEED_ENTRIES   = 100
REQUEST_DELAY_SECONDS     = 0.25
MIN_TEXT_LENGTH           = 50
MAX_UNIQUE_URLS_TO_SCRAPE = 8000
TARGET_ARTICLES_MIN       = 2000
TARGET_ARTICLES_MAX       = 6000

OUTPUT_JSON          = 'ai_companionship_news.json'
OUTPUT_CSV           = 'ai_companionship_news.csv'
OUTPUT_SENTIMENT_CSV = 'ai_companionship_news_sentiment.csv'
FAILED_OUTPUT_CSV    = 'ai_companionship_news_failed.csv'
DISCOVERY_LOG_CSV    = 'ai_companionship_discovery_log.csv'
TERM_FREQ_CSV        = 'ai_companionship_term_frequency.csv'
LDA_MODEL_SCORES_CSV = 'ai_companionship_lda_k4_k8_scores.csv'
LDA_TOPIC_TERMS_CSV  = 'ai_companionship_lda_topic_terms.csv'
SNA_CENTRALITY_CSV   = 'ai_companionship_network_centrality.csv'

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'en-US,en;q=0.9'
}

session = requests.Session()
retry_strategy = Retry(
    total=2,
    connect=2,
    read=2,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=['GET']
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount('http://', adapter)
session.mount('https://', adapter)
session.headers.update(HEADERS)

print(f'Configured {len(SEARCH_QUERIES)} search queries')
print(f'Configured {len(QUERY_RSS_SOURCES)} RSS search sources')
print(f'Configured {len(DIRECT_DISCOVERY_FEEDS)} direct RSS feeds')
print(f'Global relevance keywords: {len(GLOBAL_RELEVANCE_KEYWORDS)}')
print(f'Target article range: {TARGET_ARTICLES_MIN} to {TARGET_ARTICLES_MAX}')

Configured 68 search queries
Configured 2 RSS search sources
Configured 24 direct RSS feeds
Global relevance keywords: 33
Target article range: 2000 to 6000


## Helper Functions

These functions keep the crawl logic modular and easier to maintain. They handle URL construction, date parsing, redirect cleanup, metadata extraction, text normalization, and relevance checks.

Read this section as the shared utility layer for the notebook. If you need to change how redirects are resolved, how text is cleaned, or how relevance is checked, this is the place to do it.

### Step 04 - Load Reusable Helper Functions

Run this cell to register the low-level helpers used everywhere else in the notebook.

**What these helpers cover:** query URL construction, feed date parsing, text cleanup, redirect resolution, metadata extraction, topic matching, and post-scrape relevance checks.

In [4]:
def build_query_rss_url(template: str, query: str) -> str:
    """Inject a URL-encoded query into an RSS template."""
    return template.format(query=quote_plus(query))


def parse_feed_date(value):
    """Convert a feed date value into a UTC ISO timestamp when possible."""
    if not value:
        return None
    try:
        dt = parsedate_to_datetime(value)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc).isoformat()
    except Exception:
        return None


def clean_text(text: str) -> str:
    """Collapse repeated whitespace and return a trimmed string."""
    if not text:
        return ''
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def extract_domain(url: str) -> str:
    """Extract a lowercase domain name from a URL."""
    try:
        return urlparse(url).netloc.lower().replace('www.', '')
    except Exception:
        return ''


def normalize_phrase(text: str) -> str:
    """Normalize free text so phrase matching is more reliable."""
    text = clean_text(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def resolve_bing_redirect(url: str) -> str:
    """Unwrap Bing redirect links to the underlying publisher URL."""
    if not url:
        return ''
    parsed = urlparse(url)
    if 'bing.com' not in parsed.netloc.lower():
        return url
    query_params = parse_qs(parsed.query)
    target_url = query_params.get('url', [''])[0]
    if target_url:
        return unquote(target_url)
    return url


def resolve_google_news_redirect(url: str, timeout: int = 20) -> str:
    """Try several strategies to unwrap Google News links."""
    if not url:
        return ''

    parsed = urlparse(url)
    host = parsed.netloc.lower()
    if 'news.google.com' not in host:
        return url

    # Strategy 1: unwrap direct query-parameter links when available.
    query_params = parse_qs(parsed.query)
    for key in ['url', 'u', 'q']:
        cand = query_params.get(key, [''])[0]
        if cand and cand.startswith('http'):
            return unquote(cand)

    # Strategy 2: follow redirects on known Google News URL patterns.
    candidates = [url]
    if '/rss/articles/' in url:
        candidates.append(url.replace('/rss/articles/', '/articles/'))

    for cand in candidates:
        try:
            resp = session.get(cand, timeout=timeout, allow_redirects=True)
            final_url = (resp.url or '').strip()
            final_host = urlparse(final_url).netloc.lower()
            if final_url and 'news.google.com' not in final_host:
                return final_url

            # Strategy 3: search the returned page for an embedded publisher URL.
            html = resp.text or ''
            decoded_matches = re.findall(r"https?%3A%2F%2F[^\"'\\s<>]+", html)
            for m in decoded_matches:
                decoded = unquote(m)
                h = urlparse(decoded).netloc.lower()
                if decoded.startswith('http') and 'google.com' not in h and 'gstatic.com' not in h:
                    return decoded

            raw_matches = re.findall(r"https?://[^\"'\\s<>]+", html)
            for m in raw_matches:
                cleaned = m.replace('\u0026', '&').replace('&amp;', '&')
                h = urlparse(cleaned).netloc.lower()
                if cleaned.startswith('http') and 'google.com' not in h and 'gstatic.com' not in h:
                    return cleaned
        except Exception:
            continue

    return url


def resolve_news_redirect(url: str) -> str:
    """Resolve known aggregator redirects before article scraping."""
    url = resolve_bing_redirect(url)
    url = resolve_google_news_redirect(url)
    return url


def extract_meta_content(soup: BeautifulSoup, *names: str) -> str:
    """Return the first matching meta tag content for the supplied names."""
    for name in names:
        tag = soup.find('meta', attrs={'property': name}) or soup.find('meta', attrs={'name': name})
        if tag and tag.get('content'):
            return clean_text(tag['content'])
    return ''


def extract_with_readability(html: str) -> str:
    """Use readability as a secondary article-text extraction strategy."""
    try:
        doc = Document(html)
        readable_html = doc.summary()
        readable_soup = BeautifulSoup(readable_html, 'html.parser')
        return clean_text(readable_soup.get_text(' ', strip=True))
    except Exception:
        return ''


def matches_query(text: str, query: str) -> bool:
    """Return True when the normalized query appears in the supplied text."""
    return normalize_phrase(query) in normalize_phrase(text)


def matches_topic(text: str, keywords: list[str]) -> bool:
    """Return True when any topic keyword appears in the supplied text."""
    text_lower = clean_text(text).lower()
    return any(kw.lower() in text_lower for kw in keywords)


def is_topic_relevant(row: dict, relevance_keywords: list[str]) -> bool:
    """Apply the final relevance screen using query, title, text, and feed metadata."""
    query_text = (row.get('query') or '').lower()
    if any(kw in query_text for kw in relevance_keywords):
        return True

    combined = ' '.join([
        row.get('title') or '',
        row.get('text') or '',
        row.get('feed_title') or '',
        row.get('feed_summary') or '',
        row.get('source_hint') or '',
    ]).lower()
    return any(kw in combined for kw in relevance_keywords)

## Feed Discovery And Article Extraction

This section defines the reusable functions that talk to RSS feeds and scrape the linked article pages. It separates discovery from extraction so the notebook can log feed coverage cleanly and handle failed article downloads without breaking the rest of the run.

### Step 05 - Load Feed Discovery Functions

Run this cell to define the functions that read query-based RSS sources and direct RSS feeds.

**Step-by-step role of this cell:**

1. Build RSS URLs for each query/source combination.
2. Parse returned feed entries.
3. Normalize feed metadata.
4. Apply keyword filtering to direct RSS feeds.
5. Create a structured discovery log for monitoring feed coverage and failures.

In [5]:
def fetch_query_source_entries(query: str, source_config: dict, max_articles: int = 10) -> tuple[list[dict], dict]:
    """Fetch entries from one query-based RSS provider and preserve discovery metadata."""
    rss_url = build_query_rss_url(source_config['template'], query)
    response = session.get(rss_url, timeout=20)
    response.raise_for_status()
    feed = feedparser.parse(response.content)
    entries = []

    for entry in feed.entries[:max_articles]:
        original_url = getattr(entry, 'link', '')
        resolved_url = resolve_news_redirect(original_url)
        entries.append({
            'query': query,
            'discovery_source': source_config['source_name'],
            'feed_title': clean_text(getattr(entry, 'title', '')),
            'feed_summary': clean_text(getattr(entry, 'summary', '')),
            'url': resolved_url,
            'original_feed_url': original_url,
            'published_at': parse_feed_date(getattr(entry, 'published', None)),
            'source_hint': (
                clean_text(getattr(entry, 'source', {}).get('title', ''))
                if getattr(entry, 'source', None)
                else source_config['source_name']
            ),
        })

    log_row = {
        'discovery_source': source_config['source_name'],
        'query': query,
        'feed_url': rss_url,
        'status': 'ok',
        'fetched_entries': len(entries),
        'error': '',
    }
    return entries, log_row


def fetch_direct_source_entries(
    queries: list[str], feed_info: dict, topic_keywords: list[str], max_articles: int = 50
) -> tuple[list[dict], dict]:
    """Fetch a direct RSS feed and keep entries that match the configured topic keywords."""
    response = session.get(feed_info['feed_url'], timeout=20)
    response.raise_for_status()
    feed = feedparser.parse(response.content)
    entries = []

    for entry in feed.entries[:max_articles]:
        title = clean_text(getattr(entry, 'title', ''))
        summary = clean_text(getattr(entry, 'summary', ''))
        combined_text = f'{title} {summary}'

        # Direct-feed items are filtered before scraping to reduce noise and cost.
        if not matches_topic(combined_text, topic_keywords):
            continue

        url = getattr(entry, 'link', '')
        entries.append({
            'query': 'direct_feed',
            'discovery_source': feed_info['source_name'],
            'feed_title': title,
            'feed_summary': summary,
            'url': url,
            'original_feed_url': url,
            'published_at': parse_feed_date(getattr(entry, 'published', None)),
            'source_hint': feed_info['source_name'],
        })

    log_row = {
        'discovery_source': feed_info['source_name'],
        'query': 'direct_feed',
        'feed_url': feed_info['feed_url'],
        'status': 'ok',
        'fetched_entries': len(entries),
        'error': '',
    }
    return entries, log_row


def fetch_all_feed_entries(
    queries: list[str],
    query_rss_sources: list[dict],
    direct_feeds: list[dict],
    topic_keywords: list[str],
    max_articles_per_query: int = 25,
    max_direct_entries: int = 50,
) -> tuple[list[dict], pd.DataFrame]:
    """Collect entries from both query-based RSS providers and direct RSS feeds."""
    entries = []
    logs = []

    # Query-driven discovery from RSS search providers.
    for query in tqdm(queries, desc='Query RSS feeds'):
        for src in query_rss_sources:
            try:
                src_entries, log = fetch_query_source_entries(query, src, max_articles=max_articles_per_query)
                entries.extend(src_entries)
                logs.append(log)
            except Exception as exc:
                logs.append({
                    'discovery_source': src['source_name'],
                    'query': query,
                    'feed_url': build_query_rss_url(src['template'], query),
                    'status': 'failed',
                    'fetched_entries': 0,
                    'error': str(exc),
                })

    # Direct-feed discovery from publisher, blog, and community feeds.
    for feed_info in tqdm(direct_feeds, desc='Direct RSS feeds'):
        try:
            src_entries, log = fetch_direct_source_entries(
                queries, feed_info, topic_keywords, max_articles=max_direct_entries
            )
            entries.extend(src_entries)
            logs.append(log)
        except Exception as exc:
            logs.append({
                'discovery_source': feed_info['source_name'],
                'query': 'direct_feed',
                'feed_url': feed_info['feed_url'],
                'status': 'failed',
                'fetched_entries': 0,
                'error': str(exc),
            })

    return entries, pd.DataFrame(logs)

### Step 06 - Load Article Extraction Logic

Run this cell to define how each discovered article URL is downloaded and cleaned.

**Step-by-step role of this cell:**

1. Resolve news-aggregator redirects.
2. Download the target page.
3. Extract article text with primary and fallback methods.
4. Collect metadata such as title, author, publish date, and image.
5. Reject pages that do not produce enough real article text.

In [6]:
def extract_article(url: str) -> dict:
    """Fetch one article URL and return the cleanest text and metadata available."""
    try:
        target_url = resolve_news_redirect(url)
        response = session.get(target_url, timeout=25, allow_redirects=True)
        response.raise_for_status()
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')

        # Primary extractor: trafilatura usually gives the cleanest article body.
        extracted_text = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=False,
            favor_precision=False,
            no_fallback=False,
        )
        text = clean_text(extracted_text or '')

        # Fallback 1: readability can recover content from harder page layouts.
        if len(text) < MIN_TEXT_LENGTH:
            readability_text = extract_with_readability(html)
            if len(readability_text) > len(text):
                text = readability_text

        # Fallback 2: plain paragraph extraction catches pages the other tools miss.
        paragraphs = [p.get_text(' ', strip=True) for p in soup.find_all('p')]
        fallback_text = clean_text(' '.join(paragraphs))
        if len(fallback_text) > len(text):
            text = fallback_text

        # Extract the metadata fields used by downstream analysis.
        meta_title = extract_meta_content(soup, 'og:title', 'twitter:title')
        html_title = clean_text(soup.title.get_text(' ', strip=True)) if soup.title else ''
        title = meta_title or html_title
        authors = extract_meta_content(soup, 'author', 'article:author', 'parsely-author')
        publish_date = extract_meta_content(
            soup, 'article:published_time', 'pubdate', 'publish-date', 'date'
        ) or None
        top_image = extract_meta_content(soup, 'og:image', 'twitter:image')

        # Reject Google News wrapper pages that do not contain the publisher article.
        google_boilerplate = 'Comprehensive up-to-date news coverage, aggregated from sources all over the world by Google News.'
        if google_boilerplate.lower() in text.lower() and 'news.google.com' in extract_domain(response.url):
            raise ValueError('Google News wrapper page detected instead of publisher article')

        # Last resort: add the description if the page body is still too short.
        if len(text) < MIN_TEXT_LENGTH:
            description = extract_meta_content(soup, 'description', 'og:description', 'twitter:description')
            text = clean_text(' '.join(part for part in [description, text] if part))

        if len(text) < MIN_TEXT_LENGTH:
            raise ValueError(f'Extracted text too short ({len(text)} chars)')

        return {
            'final_url': response.url,
            'domain': extract_domain(response.url),
            'title': clean_text(title),
            'text': text,
            'authors': authors,
            'article_publish_date': publish_date,
            'top_image': top_image,
            'status': 'ok',
            'error': '',
        }
    except Exception as exc:
        return {
            'final_url': target_url if 'target_url' in locals() else url,
            'domain': extract_domain(target_url if 'target_url' in locals() else url),
            'title': '',
            'text': '',
            'authors': '',
            'article_publish_date': None,
            'top_image': '',
            'status': 'failed',
            'error': str(exc),
        }

## Main Crawl Pipeline

This section ties the earlier utilities together into one end-to-end pipeline: discover candidate URLs, deduplicate them, scrape article bodies, and apply the final relevance filter. It returns both the article dataframe and the discovery log so the run can be audited afterward.

### Step 07 - Build The End-To-End Crawl Function

This cell defines the orchestrator that runs the full crawl from discovery to final relevance screening.

**Pipeline order inside the function:**

1. Discover candidate feed entries.
2. Remove duplicate URLs.
3. Scrape article pages.
4. Measure content length and deduplicate final outputs.
5. Apply the relevance filter.
6. Return both the article table and the discovery audit log.

In [7]:
def crawl_ai_companionship_news(
    queries: list[str],
    query_rss_sources: list[dict],
    direct_feeds: list[dict],
    topic_keywords: list[str],
    relevance_keywords: list[str],
    max_articles_per_query: int = 25,
    max_direct_entries: int = 50,
    delay_seconds: float = 1.0,
    max_unique_urls_to_scrape: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run the full crawl: discover entries, deduplicate URLs, scrape pages, and apply relevance filtering."""

    # Step 1: discover candidate entries from all configured feeds.
    feed_rows, discovery_log_df = fetch_all_feed_entries(
        queries=queries,
        query_rss_sources=query_rss_sources,
        direct_feeds=direct_feeds,
        topic_keywords=topic_keywords,
        max_articles_per_query=max_articles_per_query,
        max_direct_entries=max_direct_entries,
    )

    feed_df = pd.DataFrame(feed_rows)
    if feed_df.empty:
        print('No feed entries discovered.')
        return feed_df, discovery_log_df

    # Step 2: deduplicate discovery URLs before article scraping starts.
    feed_df['url'] = feed_df['url'].astype(str).str.strip()
    feed_df = feed_df[feed_df['url'] != ''].copy()
    n_before = len(feed_df)
    feed_df = feed_df.drop_duplicates(subset=['url']).reset_index(drop=True)
    print(f'Discovered {n_before} feed entries -> {len(feed_df)} unique URLs')

    if max_unique_urls_to_scrape and len(feed_df) > max_unique_urls_to_scrape:
        feed_df = feed_df.head(max_unique_urls_to_scrape).copy()
        print(f'Applying cap: scraping first {len(feed_df)} unique URLs')

    # Step 3: scrape each article page and attach scrape metadata.
    article_rows = []
    for row in tqdm(feed_df.to_dict(orient='records'), desc='Scraping articles'):
        article_data = extract_article(row['url'])
        article_rows.append({**row, **article_data, 'scraped_at': datetime.now(timezone.utc).isoformat()})
        time.sleep(delay_seconds)

    article_df = pd.DataFrame(article_rows)
    if article_df.empty:
        return article_df, discovery_log_df

    # Step 4: compute body length and remove final duplicate articles.
    article_df['content_length'] = article_df['text'].fillna('').str.len()
    article_df = article_df.sort_values(
        ['status', 'content_length'], ascending=[True, False]
    ).reset_index(drop=True)
    article_df = article_df.drop_duplicates(
        subset=['final_url', 'title'], keep='first'
    ).reset_index(drop=True)

    # Step 5: apply the post-scrape relevance rules.
    n_total = len(article_df)
    article_df['topic_relevant'] = article_df.apply(
        lambda r: is_topic_relevant(r.to_dict(), relevance_keywords), axis=1
    )
    n_relevant = int(article_df['topic_relevant'].sum())
    print(f'Relevance filter: {n_relevant}/{n_total} articles pass')

    return article_df, discovery_log_df

## Run The Crawl

Run this section only after the configuration and helper cells have been executed. This is the notebook step that performs the actual discovery and scraping work using the current query list, RSS sources, direct RSS feeds, keyword rules, and relevance rules.

### Step 08 - Execute The Crawl With The Current Settings

Run this cell after reviewing the configuration. It uses the current query list, RSS sources, direct RSS feeds, keywords, relevance rules, and crawl limits exactly as defined above.

**After the cell runs:** review the discovery summary, scrape status counts, and relevance breakdown before moving to the save step.

In [8]:
news_df, discovery_log_df = crawl_ai_companionship_news(
    queries=SEARCH_QUERIES,
    query_rss_sources=QUERY_RSS_SOURCES,
    direct_feeds=DIRECT_DISCOVERY_FEEDS,
    topic_keywords=TOPIC_KEYWORDS,
    relevance_keywords=GLOBAL_RELEVANCE_KEYWORDS,
    max_articles_per_query=MAX_ARTICLES_PER_QUERY,
    max_direct_entries=MAX_DIRECT_FEED_ENTRIES,
    delay_seconds=REQUEST_DELAY_SECONDS,
    max_unique_urls_to_scrape=MAX_UNIQUE_URLS_TO_SCRAPE,
)

discovery_log_df.to_csv(DISCOVERY_LOG_CSV, index=False, encoding='utf-8-sig')
print('Discovery source status:')
display(discovery_log_df.groupby('status')['fetched_entries'].agg(['count', 'sum']))

if not news_df.empty:
    print('Scrape status:')
    print(news_df['status'].value_counts(dropna=False))
    print('Relevance breakdown:')
    print(news_df['topic_relevant'].value_counts())

Query RSS feeds:   0%|          | 0/68 [00:00<?, ?it/s]

Direct RSS feeds:   0%|          | 0/24 [00:00<?, ?it/s]

Discovered 180 feed entries -> 154 unique URLs


Scraping articles:   0%|          | 0/154 [00:00<?, ?it/s]

Relevance filter: 63/154 articles pass
Discovery source status:


,count,sum
status,,
failed,9,0
ok,151,180


Scrape status:
status
failed    98
ok        56
Name: count, dtype: int64
Relevance breakdown:
topic_relevant
False    91
True     63
Name: count, dtype: int64


## Save Results

This section separates successful relevant articles from failed scrapes and non-relevant articles, then writes the final outputs to disk. Treat these saved files as the operational deliverables from the crawl stage.

### Step 09 - Save Approved Outputs And Review Crawl Volume

Run this cell to separate successful relevant articles from failures and off-topic results, then write the deliverable files to disk.

**What to check here:**

1. The number of approved articles saved to CSV and JSON.
2. The number of failed scrapes saved for QA.
3. Whether the crawl volume sits inside the target operating range.

In [9]:
# Split the crawl output into approved articles, failed scrapes, and off-topic results.
clean_news_df = news_df.loc[
    (news_df['status'] == 'ok') & (news_df['topic_relevant'] == True)
].copy()

failed_news_df = news_df.loc[news_df['status'] != 'ok'].copy()
non_relevant_df = news_df.loc[
    (news_df['status'] == 'ok') & (news_df['topic_relevant'] == False)
].copy()

# Sort the final approved dataset by feed publish time for easier review.
clean_news_df = clean_news_df.sort_values(
    'published_at', ascending=False, na_position='last'
).reset_index(drop=True)

clean_news_df.to_json(OUTPUT_JSON, orient='records', force_ascii=False, indent=2)
clean_news_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
failed_news_df.to_csv(FAILED_OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'Saved {len(clean_news_df)} relevant articles to {OUTPUT_CSV} and {OUTPUT_JSON}')
print(f'Saved {len(failed_news_df)} failed rows to {FAILED_OUTPUT_CSV}')
print(f'Filtered out {len(non_relevant_df)} non-relevant articles')

if len(clean_news_df) < TARGET_ARTICLES_MIN:
    print(f'Below target range ({TARGET_ARTICLES_MIN}-{TARGET_ARTICLES_MAX}). Increase query/feed breadth or crawl cap.')
elif len(clean_news_df) > TARGET_ARTICLES_MAX:
    print(f'Above target range ({TARGET_ARTICLES_MIN}-{TARGET_ARTICLES_MAX}). Tighten relevance keywords or crawl cap.')
else:
    print(f'In target range: {len(clean_news_df)} articles.')

clean_news_df[['query', 'title', 'domain', 'published_at', 'content_length']].head(15)

Saved 30 relevant articles to ai_companionship_news.csv and ai_companionship_news.json
Saved 98 failed rows to ai_companionship_news_failed.csv
Filtered out 26 non-relevant articles
Below target range (2000-6000). Increase query/feed breadth or crawl cap.


,query,title,domain,published_at,content_length
0,uncensored AI chat Reddit,Best Dirty Talk AI: Top 13 Sex Chatbots & Apps,washingtoncitypaper.com,2026-03-31T17:00:00+00:00,21556
1,NSFW AI chatbot,OpenAI is retreating from its NSFW chatbot plans,androidauthority.com,2026-03-26T11:45:00+00:00,2644
2,adult content AI companion,Companion chatbots not doing enough to protect kids: eSafety report | Biometric Update,biometricupdate.com,2026-03-26T11:35:00+00:00,5632
3,NSFW AI chatbot,Dutch court bans Elon Musk's AI from making NSFW content,newsbytesapp.com,2026-03-26T10:49:00+00:00,733
4,AI porn chatbot,"AI is embracing erotica, but it’s not all fun and games",yahoo.com,2026-03-26T06:22:00+00:00,9352
5,uncensored AI girlfriend,Best AI Girlfriend Apps in 2026 - Advisor,straight.com,2026-03-24T11:04:00+00:00,120240
6,sexually explicit AI chat,Teens sue Elon Musk’s AI claiming image-generator made sexually explicit images of them,aol.com,2026-03-19T16:59:00+00:00,4003
7,sexually explicit AI chat,OpenAI’s adult chat plan sparks internal backlash- Moneycontrol.com,moneycontrol.com,2026-03-16T00:03:00+00:00,5632
8,AI companion age verification adult,Porn ban: Australians must prove they are over 18 to access adult content under new laws,bbc.com,2026-03-08T21:00:00+00:00,5375
9,direct_feed,"A Sexual Sandbox? New Survey Suggests ""Comfort"" is the Biggest Attraction of Chatbot Sex - Future of Sex",futureofsex.net,2026-03-08T19:40:25+00:00,5962


## Sentiment Analysis

This final stage applies VADER sentiment scoring to the cleaned article text and stores the scored dataset as a new CSV. Run it after the save step so the sentiment output is based only on the approved article set.

### Step 10 - Score Sentiment On The Final Article Set

Run this cell last. It applies VADER sentiment scoring to the cleaned article text and attaches both polarity scores and a three-class sentiment label.

**Why this step is last:** Sentiment should be computed only on the final approved dataset so the scored export reflects the exact corpus you intend to analyze downstream.

In [10]:
analyzer = SentimentIntensityAnalyzer()


def label_sentiment(score: float) -> str:
    """Convert a VADER compound score into a three-class sentiment label."""
    if score >= 0.05:
        return 'positive'
    if score <= -0.05:
        return 'negative'
    return 'neutral'


# Score each approved article body with VADER.
sentiment_scores = clean_news_df['text'].fillna('').apply(analyzer.polarity_scores).apply(pd.Series)

# Attach the raw sentiment scores and a reporting-friendly label.
sentiment_df = pd.concat([clean_news_df, sentiment_scores], axis=1)
sentiment_df['sentiment_label'] = sentiment_df['compound'].apply(label_sentiment)
sentiment_df.to_csv(OUTPUT_SENTIMENT_CSV, index=False, encoding='utf-8-sig')

print(f'Sentiment scored {len(sentiment_df)} articles')
sentiment_df[['query', 'title', 'compound', 'sentiment_label']].head(10)

Sentiment scored 30 articles


,query,title,compound,sentiment_label
0,uncensored AI chat Reddit,Best Dirty Talk AI: Top 13 Sex Chatbots & Apps,0.9999,positive
1,NSFW AI chatbot,OpenAI is retreating from its NSFW chatbot plans,-0.6400,negative
2,adult content AI companion,Companion chatbots not doing enough to protect kids: eSafety report | Biometric Update,-0.9716,negative
3,NSFW AI chatbot,Dutch court bans Elon Musk's AI from making NSFW content,-0.1446,negative
4,AI porn chatbot,"AI is embracing erotica, but it’s not all fun and games",0.9833,positive
5,uncensored AI girlfriend,Best AI Girlfriend Apps in 2026 - Advisor,1.0000,positive
6,sexually explicit AI chat,Teens sue Elon Musk’s AI claiming image-generator made sexually explicit images of them,-0.9975,negative
7,sexually explicit AI chat,OpenAI’s adult chat plan sparks internal backlash- Moneycontrol.com,-0.7822,negative
8,AI companion age verification adult,Porn ban: Australians must prove they are over 18 to access adult content under new laws,-0.9969,negative
9,direct_feed,"A Sexual Sandbox? New Survey Suggests ""Comfort"" is the Biggest Attraction of Chatbot Sex - Future of Sex",0.9985,positive
